In [1]:
!pip install transformers datasets accelerate -q

import torch
import math
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    set_seed
)
from datasets import Dataset

set_seed(42)

In [2]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [3]:
def generate_text(model, tokenizer, prompt, max_length=100):
    model.eval()
    inputs = tokenizer.encode(prompt, return_tensors="pt")

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_length=max_length,
            temperature=0.8,
            top_k=50,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [4]:
review_prompts = [
    "This product is",
    "I bought this phone and",
    "The quality of this item"
]

print("=== BASELINE REVIEWS ===")
baseline = {}

for p in review_prompts:
    baseline[p] = generate_text(model, tokenizer, p)
    print(f"\nPrompt: {p}")
    print(f"Output: {baseline[p]}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


=== BASELINE REVIEWS ===

Prompt: This product is
Output: This product is made from high quality, lightweight stainless steel. If you are looking for something a little more durable, it's a good choice.

Laser Pouch

Not all of our laser printers are created equal. We have a laser printer that comes with all of our printer parts. These parts include our new 3D printer and a 3D printed printing service. All of our printers make laser printers, including our laser printers, using laser technology. Our laser printers are the most

Prompt: I bought this phone and
Output: I bought this phone and I have not used it on a lot of people. I have also not used it on any other people.

The screen was amazing and the sound was amazing. It was not loud. I would never use it on a tv, laptop, smartphone or other connected device in the future.

The battery life is good. The phone works great but it has so many problems.

I have been using phones that have the Snapdragon 616 processor with the 8

Promp

In [5]:
corpus = [
    "this phone has an amazing battery life and the camera quality is outstanding for the price.",
    "i bought this laptop for college and it handles all my assignments and coding projects perfectly.",
    "the sound quality of these headphones is incredible with deep bass and clear vocals.",
    "this smartwatch tracks my steps accurately and the heart rate monitor is very reliable.",
    "great wireless earbuds with noise cancellation that blocks out all background sound.",
    "the keyboard feels very comfortable for long typing sessions and the backlight is a nice touch.",
    "this portable charger saved me during travel and it charges my phone three times on a single charge.",
    "the tablet screen is bright and colorful which makes watching movies a great experience.",
    "i love this fitness tracker because it motivates me to reach my daily exercise goals.",
    "this bluetooth speaker is compact but delivers surprisingly loud and clear audio.",
    "the delivery was fast and the product was packed securely with no damage at all.",
    "excellent value for money and the build quality feels premium despite the affordable price.",
    "the customer service team was very helpful when i had questions about the product features.",
    "this camera takes stunning photos in low light and the video recording quality is very smooth.",
    "i have been using this product for three months and it still works perfectly like day one.",
    "the design is sleek and modern and it looks great on my desk next to my other gadgets.",
    "easy to set up right out of the box and the instructions were clear and simple to follow.",
    "highly recommend this product to anyone looking for quality and reliability at a fair price.",
    "the software updates keep adding new features which makes this purchase even more worthwhile.",
    "best purchase i made this year and i would definitely buy from this brand again."
]

In [6]:
dataset = Dataset.from_dict({"text": corpus})

def tokenize_fn(x):
    return tokenizer(
        x["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
split = tokenized.train_test_split(test_size=0.15, seed=42)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [7]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

training_args = TrainingArguments(
    output_dir="./gpt2-reviews",
    num_train_epochs=10,   # reduced for faster run
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="no",
    fp16=torch.cuda.is_available()
)

In [8]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    data_collator=data_collator
)

trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.811784
20,2.941562
30,1.906420
40,1.215388
50,1.256534
60,0.719734
70,0.621683
80,0.513918
90,0.526318


TrainOutput(global_step=90, training_loss=1.5014823224809435, metrics={'train_runtime': 14.5946, 'train_samples_per_second': 11.648, 'train_steps_per_second': 6.167, 'total_flos': 11104911360000.0, 'train_loss': 1.5014823224809435, 'epoch': 10.0})

In [11]:
def generate_text(model, tokenizer, prompt, max_length=100):
    model.eval()
    inputs = tokenizer.encode(prompt, return_tensors="pt")
    # Move inputs to the same device as the model
    inputs = inputs.to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_length=max_length,
            temperature=0.8,
            top_k=50,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


print("\n=== FINE-TUNED REVIEWS ===")

for p in review_prompts:
    output = generate_text(model, tokenizer, p)
    print(f"\nPrompt: {p}")
    print(f"Baseline: {baseline[p][:100]}")
    print(f"Fine-tuned: {output[:100]}")


=== FINE-TUNED REVIEWS ===

Prompt: This product is
Baseline: This product is made from high quality, lightweight stainless steel. If you are looking for somethin
Fine-tuned: This product is packed securely with no damage at all. I recommend this product to anyone looking fo

Prompt: I bought this phone and
Baseline: I bought this phone and I have not used it on a lot of people. I have also not used it on any other 
Fine-tuned: I bought this phone and it handles all my everyday tasks perfectly. The camera takes stunning photos

Prompt: The quality of this item
Baseline: The quality of this item in the item description (and if the item is already in stock) will determin
Fine-tuned: The quality of this item is outstanding for the price and the product is packed securely with little


COMPONENT II: RECIPE GENERATOR

In [12]:
tokenizer2 = GPT2Tokenizer.from_pretrained("gpt2")
model2 = GPT2LMHeadModel.from_pretrained("gpt2")

tokenizer2.pad_token = tokenizer2.eos_token
model2.config.pad_token_id = tokenizer2.eos_token_id

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
recipe_prompts = [
    "To make butter chicken",
    "For pasta carbonara",
    "To prepare chocolate cake"
]

print("\n=== BASELINE RECIPES ===")

baseline2 = {}
for p in recipe_prompts:
    baseline2[p] = generate_text(model2, tokenizer2, p)
    print(f"\nPrompt: {p}")
    print(f"Output: {baseline2[p]}")


=== BASELINE RECIPES ===

Prompt: To make butter chicken
Output: To make butter chicken in a pot of the hot water, add about 2 teaspoons of butter and a pinch of salt. Cook until the butter is just melted and the skin is tender. Add the chicken and cook 5 minutes or until the skin is tender and the skin is well-steamed. Drain the butter and scoop out about 1 tablespoon of liquid.

You will need to cook the chicken for about 30 minutes or until it is very tender and firm. You will also need to use a

Prompt: For pasta carbonara
Output: For pasta carbonara, you will need to make the correct type of pasta.

For pasta crunches, I use a piece of cardboard and a piece of paper that looks like a plastic baggie and has a hole in it. When you pull the paper out, place it on the tray and hold it over it.

Now, as soon as the tray is full, you will see the pasta crunches, especially if you are using a larger size.

Next

Prompt: To prepare chocolate cake
Output: To prepare chocolate cake, melt a

In [14]:
recipes = [
    "to make butter chicken start by marinating chicken pieces in yogurt with turmeric chili powder and garam masala for one hour.",
    "heat butter in a pan and fry onions until golden brown then add ginger garlic paste and cook for two minutes.",
    "add tomato puree and cook on low heat for ten minutes until the oil separates from the masala.",
    "add the marinated chicken and cook on medium heat for fifteen minutes until fully cooked.",
    "finish with fresh cream and kasuri methi and serve hot with naan or steamed rice.",
    "for pasta carbonara boil spaghetti in salted water until al dente and reserve half cup of pasta water.",
    "fry diced pancetta in olive oil until crispy and set aside.",
    "whisk together egg yolks parmesan cheese and black pepper in a bowl.",
    "toss the hot pasta with pancetta and remove from heat then quickly stir in the egg mixture.",
    "the residual heat will cook the eggs into a creamy sauce and serve immediately with extra parmesan."
]

In [15]:
dataset2 = Dataset.from_dict({"text": recipes})

tokenized2 = dataset2.map(
    lambda x: tokenizer2(
        x["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    ),
    batched=True,
    remove_columns=["text"]
)

split2 = tokenized2.train_test_split(test_size=0.15, seed=42)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [16]:
trainer2 = Trainer(
    model=model2,
    args=training_args,
    train_dataset=split2["train"],
    eval_dataset=split2["test"],
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer2, mlm=False)
)

trainer2.train()

Step,Training Loss
10,3.538309
20,2.174660
30,1.417975
40,1.148893


TrainOutput(global_step=40, training_loss=2.0699591875076293, metrics={'train_runtime': 4.8366, 'train_samples_per_second': 16.541, 'train_steps_per_second': 8.27, 'total_flos': 5225840640000.0, 'train_loss': 2.0699591875076293, 'epoch': 10.0})

In [17]:
print("\n=== FINE-TUNED RECIPES ===")

for p in recipe_prompts:
    output = generate_text(model2, tokenizer2, p)
    print(f"\nPrompt: {p}")
    print(f"Baseline: {baseline2[p][:100]}")
    print(f"Fine-tuned: {output[:100]}")


=== FINE-TUNED RECIPES ===

Prompt: To make butter chicken
Baseline: To make butter chicken in a pot of the hot water, add about 2 teaspoons of butter and a pinch of sal
Fine-tuned: To make butter chicken a quick and easy meal for your chicken, add the chicken and cook on low heat 

Prompt: For pasta carbonara
Baseline: For pasta carbonara, you will need to make the correct type of pasta.

For pasta crunches, I use a p
Fine-tuned: For pasta carbonara with marinated onions and parmesan cheese for a further five minutes until fully

Prompt: To prepare chocolate cake
Baseline: To prepare chocolate cake, melt a large egg and beat until combined. Add a few tablespoons of vanill
Fine-tuned: To prepare chocolate cake by whisking together flour, sugar and baking powder in a bowl. Add remaini
